# Hubbard–Holstein model

Cleaned and organized for reproducible execution from the repository root.


In [ ]:
## Plan (Pseudocode)
# 1. Import libraries: numpy, qutip, matplotlib
# 2. Define system parameters: k, U, g, omega0, n_boson_levels, N_trotter, total_time
# 3. Build Hilbert space:
#    - Bosonic modes: two resonators, each truncated to n_boson_levels
#    - Fermionic modes: two sites with spin-up and spin-down => 4 qubits
#    - Total Hilbert space dimension = (2^4) * (n_boson_levels^2)
# 4. Define operators:
#    - a1, a2 = qutip.destroy(n_boson_levels) on modes 1 and 2
#    - sigma_z, sigma_x on each of the 4 fermionic qubits, mapped via tensor
#    - Combine boson and qubit operators via qutip.tensor
# 5. Construct Hamiltonian terms:
#    - H_boson = omega0 * (a1.dag()*a1 + a2.dag()*a2)
#    - H_ee = U/4 * [ (sigma_z1 + sigma_z2) + sigma_z1*sigma_z2 + (sigma_z3+sigma_z4) + sigma_z3*sigma_z4 ]
#    - H_ep = g/2 * [ (sigma_z1+sigma_z2)*(a1.dag()+a1) + (sigma_z3+sigma_z4)*(a2.dag()+a2) ]
#    - H_hop = -k * (hop_terms between qubits mapping fermion hopping via Jordan-Wigner)
# 6. Split Hamiltonian into analog block H_analog = H_boson + H_ee + H_ep, digital block H_digital = H_hop
# 7. Define initial state psi0 = tensor(fermi_half_filling, boson_vacuum)
# 8. Compute exact evolution: psi_exact = qutip.propagator(H_total, t_list)
# 9. Trotter simulation:
#    - for each step: U_analog = (-1j * H_analog * dt).expm(), U_digital = (-1j * H_digital * dt).expm()
#    - psi_trot = U_digital * U_analog * psi_current
#    - record fidelity = |<psi_exact|psi_trot>|^2
# 10. Plot fidelity vs time

In [ ]:
## Implementation
import numpy as np
import matplotlib.pyplot as plt
from qutip import tensor, basis, destroy, qeye, sigmaz, sigmax, propagator, fidelity

In [ ]:
# 1. Parameters
k = 1.0
U = 8.0 * k
g = 0.1 * k
omega0 = 8.0 * k
n_boson = 8
N_trotter = 25
T_total = 1.0  # in units of 1/k

dt = T_total / N_trotter

def tensor_ops(op_list):
    """Helper: tensor of operators list"""
    result = op_list[0]
    for op in op_list[1:]:
        result = tensor(result, op)
    return result

In [ ]:
# 2. Operators
# Boson modes
a1 = tensor(destroy(n_boson), qeye(n_boson), qeye(2), qeye(2), qeye(2), qeye(2))
a2 = tensor(qeye(n_boson), destroy(n_boson), qeye(2), qeye(2), qeye(2), qeye(2))

In [ ]:
# Qubits: mapping order [b1_up, b1_down, b2_up, b2_down]
sig_z = sigmaz()
sig_x = sigmax()
q_ops = []
for i in range(4):
    ops = []
    # two boson identity
    for _ in range(2): ops.append(qeye(n_boson))
    # qubit identities until i
    for j in range(4):
        ops.append(sig_z if j == i else qeye(2))
    # collect
    q_ops.append(tensor_ops(ops))

In [ ]:
# H_boson
H_b = omega0 * (a1.dag() * a1 + a2.dag() * a2)
# H_ee: on-site repulsion for each site
# site 1: spins 1 and 2; site 2: spins 3 and 4
H_ee = U / 4 * ((q_ops[0] + q_ops[1]) + q_ops[0] * q_ops[1] + (q_ops[2] + q_ops[3]) + q_ops[2] * q_ops[3])
# H_ep: electron-phonon coupling
H_ep = g / 2 * ((q_ops[0] + q_ops[1]) * (a1.dag() + a1) + (q_ops[2] + q_ops[3]) * (a2.dag() + a2))

In [ ]:
# H_hop: fermion hopping
# hopping between b1_down (qubit 1) and b2_up (qubit 2), and b1_up (0) to b2_down (3)
# Using sigma_x^i sigma_x^j + sigma_y^i sigma_y^j term
from qutip import sigmay
# map spin indices: up1=0, down1=1, up2=2, down2=3
pairs = [(1, 2), (0, 3)]
H_hop = 0
for (i, j) in pairs:
    # build Sx_i Sx_j + Sy_i Sy_j
    # build Sx_i
    ops_xx = []
    for _ in range(2): ops_xx.append(qeye(n_boson))
    for k in range(4): ops_xx.append(sigmax() if k == i else (sigmax() if k == j else qeye(2)))
    H_hop += -k * tensor_ops(ops_xx)
    # Similarly for Sy Sy
    ops_yy = []
    for _ in range(2): ops_yy.append(qeye(n_boson))
    for k in range(4): ops_yy.append(sigmay() if k == i or k == j else qeye(2))
    H_hop += -k * tensor_ops(ops_yy)

In [ ]:
# Total Hamiltonian for exact evolution
H_total = H_b + H_ee + H_ep + H_hop

In [ ]:
# Initial state: half-filling: |0,1,1,0> and bosons vacuum
# fermionic state
ferm = tensor(basis(2,0), basis(2,1), basis(2,1), basis(2,0))
boson = tensor(basis(n_boson, 0), basis(n_boson, 0))
psi0 = tensor(boson, ferm)

In [ ]:
# Exact evolutions
times = np.linspace(0, T_total, N_trotter+1)
U_exact = [propagator(H_total, t) for t in times]
psi_exact = [U_exact[n] * psi0 for n in range(len(times))]

In [ ]:
# Trotter simulation
psi_trot = psi0
fid_list = [1.0]
for n in range(1, N_trotter+1):
    # analog block
    U_analog = (-1j * (H_b + H_ee + H_ep) * dt).expm()
    # digital block
    U_digital = (-1j * H_hop * dt).expm()
    psi_trot = U_digital * (U_analog * psi_trot)
    fid_list.append(fidelity(psi_exact[n], psi_trot))


In [ ]:
# Plot

plt.figure()
plt.plot(times, fid_list, marker='o')
plt.xlabel('Time (1/k)')
plt.ylabel('Fidelity')
plt.title('DAQC Trotter Fidelity vs Exact')
plt.show()